# crossmatch_and_make_catalog.ipynb — HETDEX PDR1 satellite identification and catalog production

Matches each satellite streak in `intermediate/HETDEX_PDR1_sats.fits` to a
catalogued object by propagating archival two-line element sets (TLEs) with
SGP4, then writes the final publication catalog.

## What this notebook does

| § | Step | Output |
|---|------|--------|
| 1a | Load streak catalog and check TLE cache coverage | coverage report |
| 1b | Fetch missing TLE nights from Space-Track (skipped if cache is complete) | `crossmatch/tle_cache/` |
| 2 | Configure SGP4 matcher and parallelism | — |
| 3 | Run full SGP4 match over all 370 cached nights | `HETDEX_PDR1_sats_matched.fits` |
| 4 | Stamp `id_source = spacetrack` on matched rows | updated MATCH HDU |
| 5a | Query IAU CPS SatChecker for unmatched streaks (optional, ~10 min) | `satchecker_unmatched.csv` |
| 5b | Re-score SatChecker pairs with our own SGP4 scorer | updated MATCH HDU |
| 6 | Verify merged catalog | counts by id_source |
| 7 | Write publication table + spectra + CANDIDATES to final catalog | `HETDEX_PDR1_satellites.fits`, `.txt`, `.csv` |

## Outputs

| File | Description |
|------|-------------|
| `HETDEX_PDR1_satellites.fits` | Final catalog: 527 streaks, 446 identified (84.6%). Includes WAVE/SPECTRA/ERRORS and CANDIDATES HDUs — fully self-contained. |
| `HETDEX_PDR1_satellites.txt` | AAS machine-readable text (MRT format) for journal submission. |
| `HETDEX_PDR1_satellites.csv` | Plain CSV for general use. |
| `HETDEX_PDR1_sats_matched.fits` | Intermediate: full match table + candidate list. Not committed to git. |

## How to reproduce

**Restart & Run All** — or run cells in order, skipping §5a–5b if
`RUN_UNMATCHED` is not needed (saves ~10 min).

Typical runtimes:
- SGP4 match (§3): ~15 min on 10 cores
- SatChecker query (§5a): ~10 min, network-bound

## Prerequisites

```
pip install sgp4 joblib
```

[Space-Track](https://www.space-track.org) credentials in `~/.spacetrack.ini`:
```ini
[spacetrack]
identity = you@example.com
password = yourpassword
```
The TLE cache (`crossmatch/tle_cache/`, ~1.5 GB) is populated automatically
on first run. Subsequent runs skip nights already cached.

## References

- Space-Track archival GP history: https://www.space-track.org
- SGP4 propagator: Vallado et al. 2006; `sgp4` Python package
- IAU CPS SatChecker: Bassa et al. 2022 (arXiv:2408.16026)
- HETDEX PDR1: Mentuch Cooper et al. 2026, ApJS, 284, 67

In [ ]:
import os, sys, time
import numpy as np
from astropy.table import Table, hstack
from astropy.io import fits as _fits

# Pipeline modules live in crossmatch/; add it to the path.
sys.path.insert(0, os.path.join(os.path.abspath("."), "crossmatch"))

import fetch_tles as F
import match_streaks as M
from satstreak_core import MatchConfig

CATALOG    = "intermediate/HETDEX_PDR1_sats.fits"  # input streak catalog
CACHE_DIR  = "crossmatch/tle_cache"                # TLE cache stays inside crossmatch/
OUT        = "HETDEX_PDR1_sats_matched.fits"       # intermediate (not committed)

assert os.path.exists(CATALOG), f"catalog not found: {os.path.abspath(CATALOG)}"
print("catalog:", os.path.abspath(CATALOG))
print("cache  :", os.path.abspath(CACHE_DIR))

site = M.read_site(CATALOG)
print("HET site (lat, lon, elev):", site)
assert 30 < site[0] < 31 and -105 < site[1] < -103, \
    "site looks transposed: expect (30.68, -104.01, 2026)"

In [ ]:
# Load the streak catalog
info = Table.read(CATALOG, hdu="INFO")
print(f"{len(info)} streaks across {len(set(info['shotid'].tolist()))} shots")

In [ ]:
# Check TLE cache coverage — a night with no cached elements gives unmatched streaks,
# indistinguishable from 'no satellite found'.  Must be complete before matching.
coverage = M.cache_coverage(info, CACHE_DIR)

In [ ]:
# Fetch any missing TLE nights from Space-Track (incremental, skips cached nights).
# Set FETCH_LIMIT to an integer to cap the session; None fetches everything missing.
FETCH_LIMIT = None

if coverage["n_missing"] == 0:
    print("cache complete — nothing to fetch")
else:
    argv = ["--catalog", CATALOG, "--cache-dir", CACHE_DIR]
    if FETCH_LIMIT:
        argv += ["--max-nights", str(int(FETCH_LIMIT))]
    F.main(argv)
    coverage = M.cache_coverage(info, CACHE_DIR)

In [ ]:
# Match configuration and parallelism
import multiprocessing as mp

cfg = MatchConfig(
    coarse_radius_deg = 5.0,
    coarse_step_s     = 20.0,
    fine_step_s       = 0.5,
    max_perp_arcsec   = 900.0,
    max_pa_deg        = 6.0,
    exposure_span_s   = 1320.0,   # 22-minute shot; mjd_shot is the start
    margin_before_s   = 60.0,
    margin_after_s    = 60.0,
    require_sunlit    = False,
    keep_n_candidates = 5,
)

N_CORES = mp.cpu_count()
N_JOBS  = max(1, N_CORES - 2)
print(f"{N_CORES} cores detected, using n_jobs = {N_JOBS}")

In [ ]:
# Full SGP4 match over all 370 cached nights (~15 min on 10 cores).
# FORCE_MATCH = True always re-runs; set False to load from disk if OUT is current.
# NOTE: after §5b patches SatChecker rows, do NOT set FORCE_MATCH = True again
# without also re-running §5b — it would overwrite the merged results.
FORCE_MATCH = True

if M.output_is_current(OUT, CACHE_DIR, CATALOG, cfg=cfg) and not FORCE_MATCH:
    match = Table.read(OUT, hdu="MATCH")
    print(f"loaded {OUT} (up to date)")
else:
    jobs    = M.build_jobs(info, CACHE_DIR, cached_only=True)
    results = M.run_all(jobs, site, cfg, CACHE_DIR, n_jobs=N_JOBS)
    match   = M.write_output(OUT, info, results, cfg, CATALOG)

M.summarise(match)

In [ ]:
# Stamp id_source = 'spacetrack' on all matched rows (idempotent).
# §5b will overwrite this to 'satchecker' for its rows.
match = Table.read(OUT, hdu="MATCH")
if "id_source" not in match.colnames:
    match["id_source"] = np.where(np.asarray(match["matched"], bool),
                                   "spacetrack", "").astype("U12")
    with _fits.open(OUT, mode="update") as hdul:
        hdu = _fits.table_to_hdu(match)
        hdu.name = "MATCH"
        for k, h in enumerate(hdul):
            if h.name == "MATCH":
                hdul[k] = hdu
                break
        hdul.flush()
    print(f"added id_source to {OUT}")
else:
    print("id_source already present")

vals, cnts = np.unique(match["id_source"], return_counts=True)
for v, c in zip(vals, cnts):
    print(f"  {v or '(unmatched)':>12}: {c}")

In [ ]:
# Query SatChecker for every unmatched-but-searched streak.
# SatChecker's own TLE archive fills gaps in our per-night cache.
# This must run AFTER the full match (§ above) and BEFORE any cross-check
# that should remain independent of these results.
# Runtime: ~2 s per streak + polling; budget ~10 min for 80 unmatched.
# Skipped automatically if the output CSV already exists.
SC_CSV = "crossmatch/satchecker_unmatched.csv"

if os.path.exists(SC_CSV):
    print(f"{SC_CSV} already exists — skipping query")
    print("Delete the file and re-run this cell to force a fresh query.")
else:
    import satchecker_crosscheck as SC
    SC.main(["--catalog", CATALOG,
             "--matched", OUT,
             "--n-sample", "999",
             "--out",     SC_CSV,
             "--sleep",   "2.0",
             "--unmatched"])

In [ ]:
# Re-run our own SGP4 fine-pass scorer for each SatChecker-found pair,
# then write the results back into OUT with id_source = 'satchecker'.
import pandas as pd, importlib
importlib.reload(M)

chk_un = pd.read_csv(SC_CSV)
found  = chk_un[chk_un["verdict"] == "found"]
pairs  = list(zip(found["streak_id"].astype(int),
                  found["satchecker_norad"].astype(int)))
print(f"{len(pairs)} SatChecker-found streaks to rematch: {[s for s, _ in pairs]}")

new_results = M.rematch_by_norad(pairs, info, CACHE_DIR, site, cfg)
print(f"{len(new_results)} pairs scored")

_PACK_COLS = [
    "norad_id", "object_name", "object_id", "object_type", "country",
    "launch_date", "rcs_size", "constellation", "orbit_class", "illum_label",
    "match_perp_arcsec", "match_sep_arcsec", "match_pa_diff_deg",
    "match_end_a_arcsec", "match_end_b_arcsec", "match_score",
    "model_pa_deg", "streak_pa_sph_deg", "crossing_dt_s",
    "tle_age_hours", "range_km", "sat_height_km", "alt_deg",
    "sun_alt_deg", "phase_angle_deg", "ang_rate_arcsec_s",
    "ang_rate_deg_s", "t_cross_s", "g_mag_inst", "g_mag_inst_550km",
    "perigee_km", "apogee_km", "inclination_deg", "eccentricity",
    "period_min", "second_score", "score_margin", "match_time_offset_s",
    "crossing_mjd", "tle_epoch_mjd",
    "n_propagated", "n_close", "n_candidates", "illum_state", "second_norad",
    "unambiguous", "at_window_edge",
]

match_upd  = Table.read(OUT, hdu="MATCH")
streak_ids = np.asarray(match_upd["streak_id"], dtype=int)

if "id_source" not in match_upd.colnames:
    match_upd["id_source"] = np.where(
        np.asarray(match_upd["matched"], bool), "spacetrack", "").astype("U12")

for res in new_results:
    sid  = int(res["streak_id"])
    idxs = np.where(streak_ids == sid)[0]
    if not len(idxs):
        print(f"  WARNING: streak {sid} not in MATCH table — skipped")
        continue
    i = idxs[0]
    for col in _PACK_COLS:
        if col not in match_upd.colnames or col not in res:
            continue
        try:
            match_upd[col][i] = res[col]
        except (TypeError, ValueError) as e:
            print(f"  could not set {col} for streak {sid}: {e}")
    match_upd["matched"][i]   = int(res.get("norad_id", -1)) > 0
    match_upd["id_source"][i] = "satchecker"

with _fits.open(OUT, mode="update") as hdul:
    new_hdu      = _fits.table_to_hdu(match_upd)
    new_hdu.name = "MATCH"
    for k, h in enumerate(hdul):
        if h.name == "MATCH":
            hdul[k] = new_hdu
            break
    hdul.flush()

match = Table.read(OUT, hdu="MATCH")
print(f"\nUpdated {OUT}")
M.summarise(match)

In [ ]:
# Verify the merged catalog before writing the publication table.
match   = Table.read(OUT, hdu="MATCH")
_src     = np.array([s.decode() if isinstance(s, bytes) else str(s)
                     for s in match["id_source"]])
_matched = np.asarray(match["matched"], bool)

print(f"Total streaks  : {len(match)}")
print(f"Matched        : {_matched.sum()}  ({100*_matched.mean():.1f}%)")
print(f"  space-track  : {((_src == 'spacetrack') & _matched).sum()}")
print(f"  satchecker   : {((_src == 'satchecker') & _matched).sum()}")
print(f"Unmatched      : {(~_matched).sum()}")
assert _matched.sum() > 438, "SatChecker patches missing — re-run the two cells above"

In [ ]:
# Write the publication table: FITS + MRT + CSV, all 527 streaks.
# Identification columns are masked (blank) for unmatched rows — no sentinel -1 values.
if not any(c[1] == "id_source" for c in M.PUB_COLUMNS):
    M.PUB_COLUMNS.append(("M", "id_source", "IDsrc", "",
                          "Source of the identification: spacetrack or satchecker"))

pub = M.write_publication_table(
    CATALOG, OUT, out_base="HETDEX_PDR1_satellites")

_has_norad = ~pub["NORAD"].mask if hasattr(pub["NORAD"], "mask") else pub["NORAD"] > 0
print(f"Publication table: {len(pub)} streaks "
      f"({int(_has_norad.sum())} identified, {int((~_has_norad).sum())} unmatched)")

out_fits = "HETDEX_PDR1_satellites.fits"

# Append WAVE / SPECTRA / ERRORS from the streak catalog.
with _fits.open(CATALOG) as ch, _fits.open(out_fits, mode="update") as hdul:
    existing = {h.name for h in hdul}
    for name in ["WAVE", "SPECTRA", "ERRORS"]:
        if name in [h.name for h in ch]:
            if name in existing:
                del hdul[hdul.index_of(name)]
            hdul.append(_fits.ImageHDU(ch[name].data, name=name))
    hdul.flush()
print(f"Spectra HDUs (WAVE / SPECTRA / ERRORS) copied from {CATALOG}")

# Append CANDIDATES so the final file is self-contained (no need for the
# intermediate HETDEX_PDR1_sats_matched.fits for gallery plots or vetting).
cand_tbl = Table.read(OUT, hdu="CANDIDATES")
h_cand = _fits.table_to_hdu(cand_tbl)
h_cand.name = "CANDIDATES"
with _fits.open(out_fits, mode="update") as hdul:
    existing_names = [h.name for h in hdul]
    if "CANDIDATES" in existing_names:
        del hdul[hdul.index_of("CANDIDATES")]
    hdul.append(h_cand)
    hdul.flush()
print(f"CANDIDATES HDU copied from {OUT}")

# Confirm all three output files exist
print()
for fname in ["HETDEX_PDR1_satellites.fits",
              "HETDEX_PDR1_satellites.txt",
              "HETDEX_PDR1_satellites.csv"]:
    sz = os.path.getsize(fname) / 1e6 if os.path.exists(fname) else None
    status = f"{sz:.1f} MB" if sz is not None else "MISSING"
    print(f"  {fname:<40}  {status}")